# Fast Overfit Train + Eval Playground

Purpose: iterate quickly on small SRV training configs, train/evaluate in one place, and immediately see what improved.

Use this notebook with the `quantum` conda environment kernel.


In [ ]:
import os
import re
import shlex
import subprocess
from datetime import datetime
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# Paths
REPO_ROOT = Path('..').resolve()
TRAIN_SCRIPT = REPO_ROOT / 'scripts' / 'train_model.py'
EVAL_SCRIPT = REPO_ROOT / 'scripts' / 'evaluate_model.py'
RESULTS_DIR = REPO_ROOT / 'notebooks' / 'results' / 'overfit_train_eval_playground'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Base configs
TRAIN_BASE = 'overfit_debug_srv'
EVAL_BASE = 'overfit_debug_srv'

# Global run control
RUN_TRAINING = True
RUN_EVALUATION = True
DRY_RUN = False
SKIP_TRAIN_IF_MODEL_EXISTS = True
NUM_EVAL_SAMPLES = 128
GUIDANCE_SCALES = [0.5, 1.0, 1.5, 3.0, 5.0, 10.0]

STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
print('Repo root  :', REPO_ROOT)
print('Results dir:', RESULTS_DIR)
print('Timestamp  :', STAMP)


In [ ]:
# Quick knobs: edit here for fast iteration.
# These are merged on top of the base Hydra config `training=overfit_debug_srv`.
COMMON_TRAIN_OVERRIDES = {
    'training.training.max_samples': 512,
    'training.training.num_epochs': 200,
    'training.training.batch_size': 32,
    'training.training.enable_guidance_train': 'true',
    'training.training.guidance_train_p': 0.1,
    'training.scheduler.params.input_perturbation': 0.0,
}

# Add/remove experiments freely.
# `name` is a short label; final model dir also gets the timestamp suffix.
EXPERIMENTS = [
    {
        'name': 'base_lr3e4',
        'train_overrides': {
            'training.training.learning_rate': '3e-4',
        },
    },
    {
        'name': 'lr2e4_gdrop02',
        'train_overrides': {
            'training.training.learning_rate': '2e-4',
            'training.training.guidance_train_p': 0.2,
        },
    },
]

for e in EXPERIMENTS:
    print('-', e['name'])


In [ ]:
def run_cmd(cmd, cwd=REPO_ROOT, env=None, dry_run=False):
    printable = ' '.join(shlex.quote(str(x)) for x in cmd)
    print('
$ ' + printable)
    if dry_run:
        return 0, '', ''

    proc = subprocess.run(
        cmd,
        cwd=str(cwd),
        env=env,
        text=True,
        capture_output=True,
    )
    if proc.returncode != 0:
        print(proc.stdout[-2000:])
        print(proc.stderr[-2000:])
    return proc.returncode, proc.stdout, proc.stderr


def parse_eval_stdout(text):
    out = {
        'samples_requested': np.nan,
        'decoded_circuits': np.nan,
        'decode_failures': np.nan,
        'valid_rate': np.nan,
        'srv_exact_match': np.nan,
    }

    m = re.search(r"Samples requested:\s*(\d+)", text)
    if m:
        out['samples_requested'] = int(m.group(1))

    m = re.search(r"Decoded circuits\s*:\s*(\d+)", text)
    if m:
        out['decoded_circuits'] = int(m.group(1))

    m = re.search(r"Decode failures\s*:\s*(\d+)", text)
    if m:
        out['decode_failures'] = int(m.group(1))

    if not np.isnan(out['samples_requested']) and not np.isnan(out['decoded_circuits']):
        out['valid_rate'] = out['decoded_circuits'] / out['samples_requested']

    m = re.search(r"SRV exact-match rate\s*:\s*([0-9.]+)", text)
    if m:
        out['srv_exact_match'] = float(m.group(1))

    return out


def parse_training_losses(model_dir):
    model_dir = Path(model_dir)
    out = {
        'train_loss_last': np.nan,
        'train_loss_min': np.nan,
        'valid_loss_last': np.nan,
        'valid_loss_min': np.nan,
    }

    tr_path = model_dir / 'fit_losses.txt'
    va_path = model_dir / 'fit_valid_losses.txt'

    if tr_path.exists():
        tr = np.loadtxt(tr_path)
        tr = np.atleast_1d(tr)
        if tr.size > 0:
            out['train_loss_last'] = float(tr[-1])
            out['train_loss_min'] = float(np.min(tr))

    if va_path.exists():
        va = np.loadtxt(va_path)
        va = np.atleast_2d(va)
        if va.size > 0:
            vals = va[:, -1]
            out['valid_loss_last'] = float(vals[-1])
            out['valid_loss_min'] = float(np.min(vals))

    return out


def summarize_decode_errors(text, top_k=5):
    patt = re.compile(r"(?:RuntimeError|ValueError): .+")
    lines = [x.strip() for x in patt.findall(text)]
    if not lines:
        return ''
    c = Counter(lines)
    return ' | '.join(f"{n}x {msg}" for msg, n in c.most_common(top_k))


In [ ]:
# Train + evaluate all experiments
rows = []

for exp in EXPERIMENTS:
    exp_name = exp['name']
    model_name = f"{exp_name}_{STAMP}"
    model_dir = REPO_ROOT / 'models' / 'trained' / model_name

    train_overrides = {**COMMON_TRAIN_OVERRIDES, **exp.get('train_overrides', {})}

    # ----- Train -----
    train_status = 'skipped'
    if RUN_TRAINING:
        if SKIP_TRAIN_IF_MODEL_EXISTS and model_dir.exists():
            print(f"\n[SKIP TRAIN] {model_name} exists: {model_dir}")
            train_status = 'skipped_exists'
        else:
            train_cmd = [
                'python', str(TRAIN_SCRIPT),
                f'training={TRAIN_BASE}',
                f'training.general.model_name={model_name}',
                f'training.general.experiment_name={model_name}',
            ]
            for k, v in train_overrides.items():
                train_cmd.append(f'{k}={v}')

            rc, out, err = run_cmd(train_cmd, dry_run=DRY_RUN)
            (RESULTS_DIR / f'{model_name}_train_stdout.txt').write_text(out)
            (RESULTS_DIR / f'{model_name}_train_stderr.txt').write_text(err)
            train_status = 'ok' if rc == 0 else 'train_failed'

        if train_status == 'train_failed':
            rows.append({
                'experiment': exp_name,
                'model_name': model_name,
                'guidance_scale': np.nan,
                'status': train_status,
            })
            continue

    loss_stats = parse_training_losses(model_dir)

    # ----- Eval sweep -----
    if RUN_EVALUATION:
        for g in GUIDANCE_SCALES:
            eval_cmd = [
                'python', str(EVAL_SCRIPT),
                f'evaluation={EVAL_BASE}',
                f'evaluation.model_dir={model_dir}',
                f'evaluation.num_samples={NUM_EVAL_SAMPLES}',
                f'evaluation.model_params.guidance_scale={g}',
            ]

            rc, out, err = run_cmd(eval_cmd, dry_run=DRY_RUN)
            (RESULTS_DIR / f'{model_name}_eval_g{g}_stdout.txt').write_text(out)
            (RESULTS_DIR / f'{model_name}_eval_g{g}_stderr.txt').write_text(err)

            metrics = parse_eval_stdout(out)
            rows.append({
                'experiment': exp_name,
                'model_name': model_name,
                'guidance_scale': g,
                'status': 'ok' if rc == 0 else 'eval_failed',
                'top_decode_errors': summarize_decode_errors(err),
                **loss_stats,
                **metrics,
            })

results_df = pd.DataFrame(rows)
results_df


In [ ]:
# Save and reload helper
if not results_df.empty:
    out_csv = RESULTS_DIR / f'results_{STAMP}.csv'
    results_df.to_csv(out_csv, index=False)
    print('Saved:', out_csv)

# You can load a previous run by setting this path.
LOAD_CSV = None
if LOAD_CSV:
    results_df = pd.read_csv(LOAD_CSV)
    print('Loaded:', LOAD_CSV)


In [ ]:
# Immediate insights: leaderboard by decode validity first
if not results_df.empty:
    display_cols = [
        'experiment', 'model_name', 'guidance_scale', 'status',
        'valid_rate', 'srv_exact_match', 'decoded_circuits', 'decode_failures',
        'train_loss_last', 'train_loss_min', 'valid_loss_last', 'valid_loss_min',
        'top_decode_errors'
    ]
    display(results_df[display_cols].sort_values(['experiment', 'guidance_scale']))

    ok = results_df[results_df['status'] == 'ok'].copy()
    if not ok.empty:
        leaderboard = (
            ok.sort_values(['valid_rate', 'decoded_circuits', 'srv_exact_match'], ascending=[False, False, False])
              .groupby('experiment', as_index=False)
              .head(1)
              .sort_values(['valid_rate', 'decoded_circuits', 'srv_exact_match'], ascending=[False, False, False])
        )
        print('Best row per experiment (ranked by valid_rate -> decoded_circuits -> exact_match):')
        display(leaderboard[['experiment', 'guidance_scale', 'valid_rate', 'decoded_circuits', 'srv_exact_match', 'top_decode_errors']])


In [ ]:
# Plots
if not results_df.empty:
    ok = results_df[results_df['status'] == 'ok'].copy()
    if not ok.empty:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        for exp_name, grp in ok.groupby('experiment'):
            grp = grp.sort_values('guidance_scale')
            axes[0].plot(grp['guidance_scale'], grp['valid_rate'], marker='o', label=exp_name)
            axes[1].plot(grp['guidance_scale'], grp['srv_exact_match'], marker='o', label=exp_name)

        axes[0].set_title('Decode Valid Rate vs Guidance')
        axes[0].set_xlabel('guidance_scale')
        axes[0].set_ylabel('valid_rate')
        axes[0].set_ylim(0, 1)
        axes[0].grid(True, alpha=0.3)

        axes[1].set_title('SRV Exact Match vs Guidance (decoded subset)')
        axes[1].set_xlabel('guidance_scale')
        axes[1].set_ylabel('exact_match')
        axes[1].set_ylim(0, 1)
        axes[1].grid(True, alpha=0.3)
        axes[1].legend(loc='best', fontsize=8)

        plt.tight_layout()
        plt.show()


## Fast iteration workflow

1. Edit `COMMON_TRAIN_OVERRIDES` and `EXPERIMENTS`.
2. Run all cells top-to-bottom.
3. Focus on `valid_rate` first; use `srv_exact_match` only as secondary signal.
4. Inspect `top_decode_errors` for grammar/decoding failure patterns.
5. Repeat with small config changes.
